In [2]:
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sanikamal/rock-paper-scissors-dataset")

print("Path to dataset files:", path)

c:\Users\mehdi\anaconda3\envs\ACV\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 452M/452M [00:22<00:00, 21.0MB/s] 

Extracting files...


Path to dataset files: C:\Users\mehdi\.cache\kagglehub\datasets\sanikamal\rock-paper-scissors-dataset\versions\1


In [5]:
import cv2

cam = cv2.VideoCapture(0)

if not cam.isOpened():
    print("Error: Camera not opened")
    exit()

cv2.namedWindow("Camera")

counter = 0
current_frame = None

def mouse_click(event, x, y, flags, param):
    global counter, current_frame
    if event == cv2.EVENT_LBUTTONDOWN:
        counter += 1
        filename = f"data/scissors/scissors{counter}.png"
        cv2.imwrite(filename, current_frame)
        print(f"Saved {filename}")

cv2.setMouseCallback("Camera", mouse_click)

while True:
    ret, frame = cam.read()
    if not ret:
        print("Failed to grab frame")
        break

    current_frame = frame.copy()
    cv2.imshow("Camera", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()


Saved data/scissors/scissors1.png
Saved data/scissors/scissors2.png
Saved data/scissors/scissors3.png
Saved data/scissors/scissors4.png
Saved data/scissors/scissors5.png
Saved data/scissors/scissors6.png
Saved data/scissors/scissors7.png
Saved data/scissors/scissors8.png
Saved data/scissors/scissors9.png
Saved data/scissors/scissors10.png
Saved data/scissors/scissors11.png
Saved data/scissors/scissors12.png
Saved data/scissors/scissors13.png
Saved data/scissors/scissors14.png
Saved data/scissors/scissors15.png
Saved data/scissors/scissors16.png
Saved data/scissors/scissors17.png
Saved data/scissors/scissors18.png
Saved data/scissors/scissors19.png
Saved data/scissors/scissors20.png
Saved data/scissors/scissors21.png
Saved data/scissors/scissors22.png
Saved data/scissors/scissors23.png
Saved data/scissors/scissors24.png
Saved data/scissors/scissors25.png
Saved data/scissors/scissors26.png
Saved data/scissors/scissors27.png
Saved data/scissors/scissors28.png
Saved data/scissors/scissors2

In [7]:
Holistic = mp.solutions.holistic.Holistic
import cv2
import mediapipe as mp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def extract_landmarks(landmarks):
    return np.array([[l.x, l.y] for l in landmarks],dtype=np.float32)

In [8]:
import os
import pickle
dataset = []
target = []
folder_path_paper = "data/paper/" 
folder_path = folder_path_paper
all_files = os.listdir(folder_path)  # lists files and folders
files_only = [f for f in all_files if os.path.isfile(os.path.join(folder_path, f))]
counter = 0
L = len(files_only)
model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

for f in files_only:
    counter += 1
    print(100*counter/L,end="\r")
    image = plt.imread(folder_path+f)
    img_model = (image[..., :3] * 255).astype(np.uint8)
    model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
    results = model.process(img_model)
    if results.pose_landmarks is not None:
        ldk = extract_landmarks(results.pose_landmarks.landmark)
        ldk_temp = extract_landmarks(results.pose_landmarks.landmark).flatten()
        dataset.append(ldk_temp)

with open(folder_path+'dataset_paper.pkl', 'wb') as file:  # 'wb' = write binary
    pickle.dump(dataset, file)

In [9]:
import os
import pickle
dataset = []
target = []
folder_path_rock = "data/rock/" 
folder_path_scissors = "data/scissors/" 
folder_paths = [folder_path_rock,folder_path_scissors]
# folder_path = folder_path_paper
for folder_path in folder_paths:
    all_files = os.listdir(folder_path)  # lists files and folders
    files_only = [f for f in all_files if os.path.isfile(os.path.join(folder_path, f))]
    counter = 0
    L = len(files_only)
    model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)

    for f in files_only:
        counter += 1
        print(100*counter/L,end="\r")
        image = plt.imread(folder_path+f)
        img_model = (image[..., :3] * 255).astype(np.uint8)
        model = Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5)
        results = model.process(img_model)
        if results.pose_landmarks is not None:
            ldk = extract_landmarks(results.pose_landmarks.landmark)
            ldk_temp = extract_landmarks(results.pose_landmarks.landmark).flatten()
            dataset.append(ldk_temp)

    with open(folder_path+'dataset.pkl', 'wb') as file:  # 'wb' = write binary
        pickle.dump(dataset, file)

: 

### Loading Individual Datasets

In [11]:

import pickle
with open("data/paper/dataset_paper.pkl", "rb") as f:
    paper = pickle.load(f)
with open("data/rock/dataset_rock.pkl", "rb") as f:
    rock = pickle.load(f)
with open("data/scissors/dataset_scissors.pkl", "rb") as f:
    scissors = pickle.load(f)

In [ ]:
dataset = np.concatenate((np.array(rock),np.array(paper),np.array(scissors)),axis=0)
target = ['rock']*len(rock)+['paper']*len(paper)+['scissors']*len(scissors)
columns = []
for col in range(int(dataset.shape[1]/2)):
    columns.append(f"x{col}")
    columns.append(f"y{col}")


### Converting to DataFrame and Saving to Pickle

In [ ]:
import pandas as pd
data = pd.DataFrame(data= dataset,columns=columns,index = range(len(dataset)))
data["target"] = target 
data.to_pickle("dataset_total")

## Neural Network Model

In [ ]:
landmarks_of_interest = [15,16,17,18,19,20,21,22] # Right and Left Hanhs Landmarks
length_inputs = len(landmarks_of_interest)

In [86]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input,Dropout

# Create the model
model = Sequential([
    Input((2*length_inputs,)),
    Dense(128,activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')  # Output layer for 3 classes
])

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Use for one-hot encoded labels
    metrics=['accuracy']
)

# Model summary
model.summary()


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_24 (Dense)                │ (None, 128)            │         2,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,611 (49.26 KB)

 Trainable params: 12,611 (49.26 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
df = pd.read_pickle("dataset_total")
landmarks_of_interest = [15,16,17,18,19,20,21,22] # Right and Left Hanhs Landmarks
length_inputs = len(landmarks_of_interest)
colOI = []
for col in landmarks_of_interest:
    colOI.append(col*2)
    colOI.append(col*2+1)
colOI.append(len(df.columns)-1)
sub_df = df[df.columns[colOI]]
from sklearn.model_selection import train_test_split
X = df[df.columns[colOI[:-1]]].values
y = df[df.columns[-1]].values
dummies = pd.get_dummies(y).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, dummies, test_size=0.15, random_state=42)


In [ ]:
model.predict

In [88]:
history = model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=16,
    validation_data=(X_test, y_test)
)

Epoch 1/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4514 - loss: 1.0721 - val_accuracy: 0.4314 - val_loss: 1.0782
Epoch 2/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5139 - loss: 1.0301 - val_accuracy: 0.4314 - val_loss: 1.0624
Epoch 3/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5139 - loss: 1.0124 - val_accuracy: 0.4314 - val_loss: 1.0790
Epoch 4/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5139 - loss: 1.0166 - val_accuracy: 0.4314 - val_loss: 1.0663
Epoch 5/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5139 - loss: 1.0183 - val_accuracy: 0.4314 - val_loss: 1.0498
Epoch 6/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5208 - loss: 1.0126 - val_accuracy: 0.4314 - val_loss: 1.0412
Epoch 7/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5139 - loss: 1.0020 - val_accuracy: 0.4314 - val_loss: 1.0472
Epoch 8/200
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5174 - loss: 1.0019 - val_accuracy: 0.4314 - 

In [105]:
with open('model.pkl', 'wb') as file:  # 'wb' = write binary
        pickle.dump(model, file)

## RandomForest Model (66% Accuracy)

In [73]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
params = {'n_estimators': range(100,800,100),
          'max_depth' : range(2,5,2)}
cv = RandomizedSearchCV(rf,params,verbose=2,scoring="accuracy",n_jobs=-1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42)
history = cv.fit(X_train,y_train)


Fitting 5 folds for each of 10 candidates, totalling 50 fits


In [74]:
pd.DataFrame(history.cv_results_)


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_n_estimators,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,1.195370,0.023129,0.088505,0.006555,600,4,"{'n_estimators': 600, 'max_depth': 4}",0.706897,0.603448,0.620690,0.684211,0.614035,0.645856,0.041572,8
1,0.192483,0.021954,0.012430,0.001859,100,2,"{'n_estimators': 100, 'max_depth': 2}",0.620690,0.637931,0.689655,0.596491,0.719298,0.652813,0.045173,3
2,1.524023,0.075228,0.093853,0.009314,700,2,"{'n_estimators': 700, 'max_depth': 2}",0.620690,0.672414,0.689655,0.561404,0.701754,0.649183,0.051882,5
3,0.840734,0.050709,0.052605,0.013381,400,2,"{'n_estimators': 400, 'max_depth': 2}",0.655172,0.655172,0.706897,0.561404,0.719298,0.659589,0.055646,2
4,0.231173,0.020247,0.013568,0.004035,100,4,"{'n_estimators': 100, 'max_depth': 4}",0.724138,0.603448,0.603448,0.684211,0.596491,0.642347,0.052099,9
5,0.932329,0.055516,0.056698,0.005751,400,4,"{'n_estimators': 400, 'max_depth': 4}",0.706897,0.637931,0.637931,0.684211,0.596491,0.652692,0.038797,4
6,1.126487,0.031704,0.067970,0.007856,500,4,"{'n_estimators': 500, 'max_depth': 4}",0.706897,0.603448,0.586207,0.684211,0.614035,0.638959,0.047599,10
7,0.980709,0.092516,0.066166,0.005243,500,2,"{'n_estimators': 500, 'max_depth': 2}",0.603448,0.689655,0.706897,0.578947,0.666667,0.649123,0.049597,7
8,0.417215,0.004333,0.028461,0.001564,200,2,"{'n_estimators': 200, 'max_depth': 2}",0.603448,0.672414,0.706897,0.561404,0.701754,0.649183,0.057326,5
9,0.986080,0.029130,0.052241,0.013281,600,2,"{'n_estimators': 600, 'max_depth': 2}",0.620690,0.689655,0.706897,0.596491,0.701754,0.663097,0.045503,1


## Training XgBoost Model (66% Accuracy)

In [77]:
ybis = y
ybis[ybis=='rock'] = 0 
ybis[ybis=='paper'] = 1 
ybis[ybis=='scissors'] = 2 
X_train, X_test, y_train, y_test = train_test_split(
    X, ybis, test_size=0.15, random_state=42)

In [78]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


model = xgb.XGBClassifier(
    objective='multi:softprob',  # probabilities for each class
    num_class=3,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=True
)


[0]	validation_0-mlogloss:1.04243
[1]	validation_0-mlogloss:1.01166
[2]	validation_0-mlogloss:0.98542
[3]	validation_0-mlogloss:0.95266
[4]	validation_0-mlogloss:0.92429
[5]	validation_0-mlogloss:0.90269
[6]	validation_0-mlogloss:0.88242
[7]	validation_0-mlogloss:0.85825
[8]	validation_0-mlogloss:0.84342
[9]	validation_0-mlogloss:0.82595
[10]	validation_0-mlogloss:0.80786
[11]	validation_0-mlogloss:0.79072
[12]	validation_0-mlogloss:0.77851
[13]	validation_0-mlogloss:0.76497
[14]	validation_0-mlogloss:0.75371
[15]	validation_0-mlogloss:0.74318
[16]	validation_0-mlogloss:0.73348
[17]	validation_0-mlogloss:0.72282
[18]	validation_0-mlogloss:0.71443
[19]	validation_0-mlogloss:0.70671
[20]	validation_0-mlogloss:0.70088
[21]	validation_0-mlogloss:0.69352
[22]	validation_0-mlogloss:0.68341
[23]	validation_0-mlogloss:0.67402
[24]	validation_0-mlogloss:0.66635
[25]	validation_0-mlogloss:0.66120
[26]	validation_0-mlogloss:0.65694
[27]	validation_0-mlogloss:0.64872
[28]	validation_0-mlogloss:0.6

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [85]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
acc = accuracy_score(y_test.astype(int), y_pred)
print(f"Validation Accuracy: {acc:.4f}")


Validation Accuracy: 0.6667


## Live Test with Webcam

In [ ]:
import cv2
import mediapipe as mp
import pickle
def extract_landmarks(landmarks):
    return np.array([[l.x, l.y] for l in landmarks],dtype=np.float32)
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils
gesture = ['paper', 'rock', 'scissors','None']
cap = cv2.VideoCapture(0)  # Open webcam
with open("model.pkl", "rb") as f:
    model = pickle.load(f)
ind = 3 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Flip for a mirror view
        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Make predictions
        # results = holistic.process(rgb_frame)
        # if results.pose_landmarks is not None:
        #     features = extract_landmarks(results.pose_landmarks.landmark).reshape(1, -1)
        #     sample = features[0:1, colOI[:-1]]   # already (1, n_features)
        #     print(features.shape)
        #     # Predict using the pre-trained ML model
        #     print(sample.shape)
        #     prediction = model.predict(sample)
        #     if (prediction>.85).any():
        #         ind = np.argmax(prediction)
        #     else:
        #         ind = 3
         

        # Display prediction
        cv2.putText(frame, f'Gesture Detected: {gesture[ind]}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

        cv2.imshow("Webcam Feed", frame)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC to quit
            break

cap.release()
cv2.destroyAllWindows()


In [109]:
colOI

[30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 66]

In [110]:
a = [[0.6440072  0.41427925 0.6550845  0.3587035  0.6670046  0.36315393
  0.6774372  0.36766112 0.6089681  0.34957683 0.58764136 0.34811047
  0.56470305 0.34927654 0.6756053  0.4048103  0.5085685  0.386199
  0.6566351  0.49297535 0.59926575 0.48311532 0.7901277  0.7576076
  0.31602162 0.7294971  0.95098895 0.9955032  0.15297085 1.0430536
  1.053354   1.486327   0.12903345 1.494215   1.112877   1.5920767
  0.0828304  1.6046854  1.0716107  1.6125215  0.13509011 1.6163983
  1.035111   1.5777702  0.15915954 1.5731304  0.687822   1.5677017
  0.3951792  1.5645199  0.6724082  2.2316332  0.37960374 2.2390933
  0.6628778  2.830731   0.37456837 2.8232858  0.66386133 2.9349785
  0.3623852  2.926754   0.6291988  3.013928   0.4306473  3.0097558 ]]

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2400572819.py, line 1)